In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.svm import SVR
from xgboost import XGBRegressor
import pandas as pd

In [ ]:
df = pd.read_csv("/content/dataset.csv", encoding="utf-8")

numeric_columns = [
    "Hours_Studied",
    "Attendance",
    "Sleep_Hours",
    "Previous_Scores",
    "Tutoring_Sessions",
    "Physical_Activity",
]


ordinal_columns = {
    "columns_names": [
        "Parental_Involvement",
        "Access_to_Resources",
        "Motivation_Level",
        "Family_Income",
        "Teacher_Quality",
        "Parental_Education_Level",
        "Distance_from_Home",
    ],
    "columns_categories": [
        ["Low", "Medium", "High"],
        ["Low", "Medium", "High"],
        ["Low", "Medium", "High"],
        ["Low", "Medium", "High"],
        ["Low", "Medium", "High"],
        ["High School", "College", "Postgraduate"],
        ["Near", "Moderate", "Far"],
    ],
}

nominal_columns = [
    "Gender",
    "School_Type",
    "Extracurricular_Activities",
    "Internet_Access",
    "Learning_Disabilities",
    "Peer_Influence",
]

**Numeric** pipline

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("fill_empty_values_with_median", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

**Ordinal** pipline

In [ ]:
ordinal_pipeline = Pipeline(
    steps=[
        ("fill_empty_values_with_frequent", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OrdinalEncoder(
                categories=ordinal_columns["columns_categories"],
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

**Nominal** pipline

In [ ]:
nominal_pipeline = Pipeline(
    steps=[
        ("fill_missing_values_with_frequent", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

Preparing the processing columns pipline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("ordinal", ordinal_pipeline, ordinal_columns["columns_names"]),
        ("nominal", nominal_pipeline, nominal_columns),
    ],
    remainder="drop",
)

Remove the outliers **iqr**, same as **Z-Score**

In [ ]:
# iqr outliers
q1 = df["Exam_Score"].quantile(0.25)
q3 = df["Exam_Score"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
iqr_outlier_mask = (df["Exam_Score"] < lower_bound) | (df["Exam_Score"] > upper_bound)
removed_rows = int(iqr_outlier_mask.sum())
df = df[~iqr_outlier_mask]

spliting data for training and testing

In [ ]:
X = df.drop(columns="Exam_Score")
y = df["Exam_Score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

saving models with algorithme of training to loop trough theme in the training step

In [ ]:
models = [
    (
        "Linear Regression",
        LinearRegression(),
        {
            "regressor__fit_intercept": [True, False],
            "regressor__positive": [False, True],
        },
    ),
    ("Ridge", Ridge(), {"regressor__alpha": [0.1, 1.0, 10.0]}),
    (
        "Random Forest",
        RandomForestRegressor(random_state=42, n_jobs=-1),
        {
            "regressor__n_estimators": [100, 300],
            "regressor__max_depth": [None, 10],
            "regressor__min_samples_split": [2, 5],
        },
    ),
    (
        "XGBoost",
        XGBRegressor(random_state=42, tree_method="hist", n_jobs=-1),
        {
            "regressor__n_estimators": [100, 300],
            "regressor__learning_rate": [0.05, 0.1],
            "regressor__max_depth": [3, 6],
            "regressor__subsample": [0.8, 1.0],
        },
    ),
    (
        "SVR",
        SVR(),
        {
            "regressor__C": [1.0, 10.0],
            "regressor__kernel": ["rbf", "linear"],
            "regressor__epsilon": [0.1, 0.5],
        },
    ),
]

In [ ]:
cv = KFold(n_splits=3,shuffle=True,random_state=42)

In [ ]:
results = []
best_model = None
best_cv_score = float("-inf")
for name, regressor, param_grid in models:
    candidate = Pipeline(
        steps=[
            ("preprocessing", preprocessor),
            ("regressor", regressor),
        ]
    )
    print(f"Tuning {name} with 3-fold cross-validation...", flush=True)
    search = GridSearchCV(
        estimator=candidate,
        param_grid=param_grid,
        scoring="r2",
        cv=cv,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    candidate = search.best_estimator_
    if search.best_score_ > best_cv_score:
        best_model = candidate
        best_cv_score = search.best_score_

Tuning Linear Regression with 3-fold cross-validation...
Tuning Ridge with 3-fold cross-validation...
Tuning Random Forest with 3-fold cross-validation...
Tuning XGBoost with 3-fold cross-validation...
Tuning SVR with 3-fold cross-validation...


In [ ]:
print(best_model.named_steps['regressor'])

SVR(C=10.0, epsilon=0.5, kernel='linear')
